In [1]:
from kaldo.forceconstants import ForceConstants
from kaldo.cumulant.api import cumulant_thermo
from kaldo.cumulant.sampler import SCSampler
import numpy as np


2026-05-28 15:15:22.388087: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-28 15:15:22.496969: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-28 15:15:24.209167: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
ifc_basepath = "/home/emeitz/software/kaldo/kaldo/tests/cumulant_fixtures/SW/100K_3UC"
ifc_tdep = ForceConstants.from_folder(
        ifc_basepath,
        supercell = (3,3,3),
        format="tdep",
        include_fourth = True
    )

(108, 1, 1)


In [3]:
sc = ifc_tdep.second.replicated_atoms
species_sc = sc.get_chemical_symbols()
masses_amu_sc = sc.get_masses()
n_sc = len(sc)

ifc2_remapped_raw = ifc_tdep.irred_to_full(order = 2)
print(ifc2_remapped_raw.shape)
ifc2_remapped_raw = ifc2_remapped_raw.reshape((3 * n_sc,3 * n_sc))
ifc2_remapped_raw.shape

(108, 2, 3, 108, 2, 3)


(648, 648)

In [4]:
print(ifc2_remapped_raw[:3,3:6])
print(np.sum(ifc2_remapped_raw))

[[-4.54802472 -2.87196668 -2.87196668]
 [-2.87196668 -4.54802472 -2.87196668]
 [-2.87196668 -2.87196668 -4.54802472]]
-5.571765271383811e-12


In [5]:
sampler = SCSampler(
    ifc2_remapped_raw,
    masses_amu_sc,
    T_K=100,
    is_classic=True,
    seed=1234
)

[0.         0.         0.         0.00044979 0.00044979 0.00044979
 0.00044979 0.00044979 0.00044979 0.00044979]


In [6]:
ifc_tdep.second.replicated_atoms.cell

Cell([[293.21999999999997, 293.21999999999997, 0.0], [0.0, 2.715, 2.715], [2.715, 0.0, 2.715]])

In [8]:
T = 100
is_classic = True
sw_path = "./kaldo/tests/cumulant_fixtures/SW/100K_3UC/Si.sw"
pot_cmds = ["pair_style sw", f"pair_coeff * * \"{sw_path}\" Si"]
cumulant_thermo(
    ifc_basepath,
    (3,3,3),
    T,
    is_classic,
    pot_cmds,
    1_000,
    10,
    harmonic_mesh = (30, 30, 30),
    free_energy_mesh = (15,15,15),
    verbose = True
)

(108, 1, 1)
Parsed unit-cell species with ASE as: ['Si', 'Si']
Parsed unit-cell masses with ASE as: [28.085 28.085]
Phase 1: harmonic (30, 30, 30) ...
  F_H=+6.9668e-02  U_H=+7.2856e-02  S_H=+3.6991e-01  Cv_H=+7.4064e-01  (10.6s)
Phase 5: sampling N=1000 configs ...
Phase 5: Remapped IFCs to supercell ...
IFC2 shape: (648, 648)
[0.         0.         0.         0.00044979 0.00044979 0.00044979
 0.00044979 0.00044979 0.00044979 0.00044979]
Phase 5: Built sampler ...
Phase 5: Build contractors ...
Phase 5: Building LAMMPS calculator ...
  LAMMPS atom_types: {'Si': 1}
  LAMMPS atom_type_masses: {'Si': 28.085}


 10%|▉         | 99/1000 [00:26<04:00,  3.75it/s]

  n=100/1000  (26.7s)


 20%|█▉        | 199/1000 [00:53<03:35,  3.71it/s]

  n=200/1000  (53.5s)


...

100%|█████████▉| 999/1000 [04:26<00:00,  3.76it/s]

  n=1000/1000  (267.2s)


100%|██████████| 1000/1000 [04:27<00:00,  3.74it/s]

F0=+6.5211e+13 +- 4.56e+13  U0=-6.5285e+15 +- 4.62e+15  S0=-7.6517e+17 +- 5.42e+17  Cv0=-2.2207e+19 +- 1.76e+19


NameError: name 'res1' is not defined